In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.ensemble import IsolationForest

from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    roc_auc_score,
    average_precision_score
)

In [2]:
X_train = pd.read_csv("../data/processed/X_train.csv")
X_val = pd.read_csv("../data/processed/X_val.csv")
X_test = pd.read_csv("../data/processed/X_test.csv")

train_metadata = pd.read_csv("../data/processed/train_metadata.csv")
val_metadata = pd.read_csv("../data/processed/val_metadata.csv")
test_metadata = pd.read_csv("../data/processed/test_metadata.csv")

print("Training set:", X_train.shape)
print("Validation set:", X_val.shape)
print("Test set:", X_test.shape)

Training set: (7209, 92)
Validation set: (55356, 92)
Test set: (55357, 92)


In [3]:
y_val = (val_metadata["Label"] == "Anomaly").astype(int)
y_test = (test_metadata["Label"] == "Anomaly").astype(int)

print("Validation labels:")
print(y_val.value_counts())

print("\nTest labels:")
print(y_test.value_counts())

Validation labels:
Label
1    52953
0     2403
Name: count, dtype: int64

Test labels:
Label
1    52954
0     2403
Name: count, dtype: int64


In [4]:
baseline_model = IsolationForest(
    random_state = 42
)

baseline_model.fit(X_train)

IsolationForest(random_state=42)

In [5]:
baseline_predictions = baseline_model.predict(X_val)

baseline_predictions = (
    baseline_predictions == -1
).astype(int)

In [6]:
cm = confusion_matrix(y_val, baseline_predictions)
print(cm)

tn, fp, fn, tp = cm.ravel()

print("\nTrue Negatives:", tn)
print("False Positives:", fp)
print("False Negatives:", fn)
print("True Positives:", tp)

[[ 2183   220]
 [ 1366 51587]]

True Negatives: 2183
False Positives: 220
False Negatives: 1366
True Positives: 51587


In [7]:
accuracy = accuracy_score(y_val, baseline_predictions)
precision = precision_score(y_val, baseline_predictions)
recall = recall_score(y_val, baseline_predictions)
f1 = f1_score(y_val, baseline_predictions)

balanced_accuracy = balanced_accuracy_score(
    y_val,
    baseline_predictions
)

fpr = fp / (fp + tn)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1-score:", f1)
print("Balanced Accuracy:", balanced_accuracy)
print("False Positive Rate:", fpr)

Accuracy: 0.9713490859166125
Precision: 0.9957534696083541
Recall: 0.974203538987404
F1-score: 0.984860633829706
Balanced Accuracy: 0.9413256563018584
False Positive Rate: 0.09155222638368705


In [8]:
baseline_scores = -baseline_model.decision_function(X_val)

roc_auc = roc_auc_score(
    y_val,
    baseline_scores
)

pr_auc = average_precision_score(
    y_val,
    baseline_scores
)

print("ROC-AUC:", roc_auc)
print("PR-AUC:", pr_auc)

ROC-AUC: 0.9645524581629675
PR-AUC: 0.9951903021250628


In [9]:
contamination_values = [
    "auto",
    0.001,
    0.005,
    0.01,
    0.02,
    0.05,
    0.10
]

In [10]:
contamination_results = []

for contamination in contamination_values:
    
    print("Testing contamination =", contamination)
    
    model = IsolationForest(
        n_estimators = 100,
        contamination = contamination,
        random_state = 42,
        n_jobs = 1
    )
    
    model.fit(X_train)
    
    predictions = model.predict(X_val)
    predictions = (predictions == -1).astype(int)
    
    cm = confusion_matrix(y_val, predictions)
    tn, fp, fn, tp = cm.ravel()
    
    precision = precision_score(y_val, predictions)
    recall = recall_score(y_val, predictions)
    f1 = f1_score(y_val, predictions)
    
    balanced_accuracy = balanced_accuracy_score(
        y_val,
        predictions
    )
    
    fpr = fp / (fp + tn)
    
    scores = -model.decision_function(X_val)
    
    roc_auc = roc_auc_score(y_val, scores)
    pr_auc = average_precision_score(y_val, scores)
    
    contamination_results.append({
        "contamination": contamination,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "balanced_accuracy": balanced_accuracy,
        "fpr": fpr,
        "roc_auc": roc_auc,
        "pr_auc": pr_auc
    })

Testing contamination = auto
Testing contamination = 0.001
Testing contamination = 0.005
Testing contamination = 0.01
Testing contamination = 0.02
Testing contamination = 0.05
Testing contamination = 0.1


In [ ]:
contamination_df = pd.DataFrame(
    contamination_results
)

contamination_df

contamination_df = contamination_df.sort_values(
    by = "balanced_accuracy",
    ascending = False
)

contamination_df

best_contamination_result = contamination_df.iloc[0]

print(
    "Best contamination:",
    best_contamination_result["contamination"]
)

print(
    "Balanced Accuracy:",
    best_contamination_result["balanced_accuracy"]
)

print(
    "Recall:",
    best_contamination_result["recall"]
)

print(
    "False Positive Rate:",
    best_contamination_result["fpr"]
)

In [ ]:
numeric_contamination = contamination_df[
    contamination_df["contamination"] != "auto"
].copy()

numeric_contamination["contamination"] = (
    numeric_contamination["contamination"].astype(float)
)

numeric_contamination = numeric_contamination.sort_values(
    "contamination"
)

plt.figure(figsize=(7,5))

plt.plot(
    numeric_contamination["contamination"],
    numeric_contamination["balanced_accuracy"],
    marker="o"
)

plt.xlabel("Contamination")
plt.ylabel("Balanced Accuracy")
plt.title("Effect of contamination on Isolation Forest")
plt.grid()

plt.show()

In [ ]:
plt.figure(figsize=(7,5))

plt.scatter(
    contamination_df["fpr"],
    contamination_df["recall"]
)

plt.xlabel("False Positive Rate")
plt.ylabel("Recall")
plt.title("Isolation Forest - Recall vs False Positive Rate")
plt.grid()

plt.show()

In [ ]:
n_estimators_values = [50, 100, 200, 300]

best_contamination = contamination_df.iloc[0][
    "contamination"
]

print(
    "Selected contamination:",
    best_contamination
)

In [ ]:
estimators_results = []

for n_estimators in n_estimators_values:
    
    print(
        "Testing n_estimators =",
        n_estimators
    )

model = IsolationForest(
    n_estimators = n_estimators,
    contamination = best_contamination,
    random_state = 42,
    n_jobs = -1
)

model.fit(X_train)

predictions = model.predict(X_val)
predictions = (predictions == -1).astype(int)   

cm = confusion_matrix(y_val, predictions) 
tn, fp, fn, tp = cm.ravel()

precision = precision_score(
    y_val,
    predictions
)

recall = recall_score(
    y_val,
    predictions
)

f1 = f1_score(
    y_val,
    predictions
)

balanced_accuracy = balanced_accuracy_score(
    y_val,
    predictions
)

fpr = fp / (fp + tn)

scores = -model.decision_function(X_val)

roc_auc = roc_auc_score(
    
    y_val,
    scores
)

pr_auc = average_precision_score(
    y_val, 
    scores
)

estimators_results.append({
    "n_estimators": n_estimators,
    "balanced_accuracy": balanced_accuracy,
    "precision": precision,
    "recall": recall,
    "f1": f1,
    "fpr": fpr,
    "roc_auc": roc_auc,
    "pr_auc": pr_auc
})